In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# ── Extract both datasets ──
HEMO_ZIP   = '/content/drive/MyDrive/Datasets/hemo_dataset.zip'
VENT_ZIP   = '/content/drive/MyDrive/Datasets/vent_dataset.zip'
HEMO_DIR   = '/content/hemo_data'
VENT_DIR   = '/content/vent_data'

for zip_path, extract_dir in [(HEMO_ZIP, HEMO_DIR), (VENT_ZIP, VENT_DIR)]:
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_dir)
    print(f"✅ Extracted to {extract_dir}: {os.listdir(extract_dir)}")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## Inference Harvesting

We load the frozen CNNs, run every 1-second chunk through them, and produce an **AI thought history** — a time-series of `[hemo_prob, vent_prob]` pairs.

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 1.1 — Define CNN Architectures (must match training)
# ══════════════════════════════════════════════════════════════

class HemoScout1DCNN(nn.Module):
    """Hemo-Scout: predicts cardiovascular crash from 1s PLETH.
    Input: [batch, 1, 100]  Output: [batch] (raw logit)
    Architecture: single 'features' Sequential block."""
    def __init__(self, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(128, 1),
        )
    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)


class VentGuardian1DCNN(nn.Module):
    """Vent-Guardian: predicts respiratory crisis from 1s CO2.
    Input: [batch, 1, 100]  Output: [batch] (raw logit)

    IMPORTANT: This architecture uses named blocks (block1, block2, block3)
    with double convolutions per block. This is DIFFERENT from HemoScout's
    single-'features'-Sequential design. The structure must match the saved
    vent_guardian.pth state_dict exactly."""
    def __init__(self, dropout=0.0):
        super().__init__()
        # Block 1: catch short-term wave features (upstrokes, peaks)
        self.block1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Conv1d(32, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),                                 # 100 → 50
        )
        # Block 2: catch mid-range patterns (waveform flattening, baseline drift)
        self.block2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),                                 # 50 → 25
        )
        # Block 3: high-level compression
        self.block3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(4),                         # → 128 × 4 = 512
        )
        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x).squeeze(1)

print("✅ CNN architectures defined")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 1.2 — Load Frozen Weights
# ══════════════════════════════════════════════════════════════

HEMO_WEIGHTS = '/content/drive/MyDrive/Datasets/hemo_scout.pth'
VENT_WEIGHTS = '/content/drive/MyDrive/Datasets/vent_guardian.pth'

hemo_model = HemoScout1DCNN(dropout=0.0).to(device)  # dropout=0 for inference
hemo_model.load_state_dict(torch.load(HEMO_WEIGHTS, map_location=device))
hemo_model.eval()
print(f"✅ Hemo-Scout loaded (params: {sum(p.numel() for p in hemo_model.parameters()):,})")

vent_model = VentGuardian1DCNN(dropout=0.0).to(device)
vent_model.load_state_dict(torch.load(VENT_WEIGHTS, map_location=device))
vent_model.eval()
print(f"✅ Vent-Guardian loaded (params: {sum(p.numel() for p in vent_model.parameters()):,})")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 1.3 — Inference Harvesting
# ══════════════════════════════════════════════════════════════

hemo_X = np.load(os.path.join(HEMO_DIR, 'hemo_X_train.npy'))  # [N_hemo, 1, 100]
hemo_Y = np.load(os.path.join(HEMO_DIR, 'hemo_Y_train.npy'))  # [N_hemo]
vent_X = np.load(os.path.join(VENT_DIR, 'vent_X_train.npy'))  # [N_vent, 1, 100]
vent_Y = np.load(os.path.join(VENT_DIR, 'vent_Y_train.npy'))  # [N_vent]

print(f"Hemo data: X={hemo_X.shape}, Y={hemo_Y.shape}")
print(f"Vent data: X={vent_X.shape}, Y={vent_Y.shape}")

# ── DIAGNOSTIC: Check original CNN dataset class balance ──
print(f"\n⚠️  ORIGINAL CNN DATASET CLASS BALANCE:")
print(f"  Hemo crashes: {hemo_Y.mean():.1%}  (Safe={int((hemo_Y==0).sum()):,} / Crash={int((hemo_Y==1).sum()):,})")
print(f"  Vent crashes: {vent_Y.mean():.1%}  (Safe={int((vent_Y==0).sum()):,} / Crash={int((vent_Y==1).sum()):,})")

N = min(len(hemo_X), len(vent_X))
print(f"\nShared timeline length: {N:,} seconds")

# Compute union label balance preview
preview_labels = np.maximum(hemo_Y[:N], vent_Y[:N])
print(f"  Union crashes: {preview_labels.mean():.1%}")
print(f"  → This is the approximate Transformer label distribution.")

# Run inference in batches
BATCH = 512

def harvest_probs(model, X, n, batch_size=BATCH):
    """Run a frozen CNN over all chunks and return probability scores."""
    probs = np.zeros(n, dtype=np.float32)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            batch = torch.from_numpy(X[start:end]).to(device)
            logits = model(batch)
            probs[start:end] = logits.sigmoid().cpu().numpy()
    return probs

print("\n🔄 Harvesting Hemo-Scout predictions...")
hemo_probs = harvest_probs(hemo_model, hemo_X, N)
print(f"   Done. Range: [{hemo_probs.min():.4f}, {hemo_probs.max():.4f}]")

print("🔄 Harvesting Vent-Guardian predictions...")
vent_probs = harvest_probs(vent_model, vent_X, N)
print(f"   Done. Range: [{vent_probs.min():.4f}, {vent_probs.max():.4f}]")

# Stack into thought history: [N, 2]
thought_history = np.stack([hemo_probs, vent_probs], axis=1).astype(np.float32)
print(f"\n✅ Thought history shape: {thought_history.shape}")

## Ground Truth Labeling & Thought Sequence Chunking

The Transformer sees 60-second windows of AI predictions. Each window is labeled based on **physiological truth**: did a real crash actually occur 5 minutes after the window ended?

Since both CNNs already predict 5 minutes ahead, we use the **union** of both original label vectors: if *either* modality says a crash happens at `t + 5min`, the Transformer label is `1`.

### v2 Fixes
- **Chronological split** instead of random (avoids 98% overlap leakage)
- **Corrected `pos_weight`** — always upweight the minority class

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 2.1 — Build Ground Truth
# ══════════════════════════════════════════════════════════════

WINDOW_SIZE = 60  # 60 seconds of AI thought history

# Combine labels: crash if EITHER modality predicts crash
combined_labels = np.maximum(hemo_Y[:N], vent_Y[:N]).astype(np.int64)

# Number of possible windows
n_windows = N - WINDOW_SIZE + 1
print(f"Total possible 60-second windows: {n_windows:,}")

# Build sliding window dataset
transformer_X = np.zeros((n_windows, WINDOW_SIZE, 2), dtype=np.float32)
transformer_Y = np.zeros(n_windows, dtype=np.int64)

for i in range(n_windows):
    transformer_X[i] = thought_history[i:i + WINDOW_SIZE]
    transformer_Y[i] = combined_labels[i + WINDOW_SIZE - 1]

print(f"\nTransformer X shape: {transformer_X.shape}")
print(f"Transformer Y shape: {transformer_Y.shape}")

n_safe  = int((transformer_Y == 0).sum())
n_crash = int((transformer_Y == 1).sum())
print(f"\nClass balance:")
print(f"  Safe  (0): {n_safe:>8,} ({n_safe/len(transformer_Y)*100:.1f}%)")
print(f"  Crash (1): {n_crash:>8,} ({n_crash/len(transformer_Y)*100:.1f}%)")

# Determine which class is the minority
if n_crash < n_safe:
    minority_label, majority_label = 'Crash', 'Safe'
    pos_weight_value = n_safe / max(n_crash, 1)
else:
    minority_label, majority_label = 'Safe', 'Crash'
    # When crashes are the majority, pos_weight < 1 DOWNweights the positive class.
    # This is actually correct behavior for BCEWithLogitsLoss pos_weight.
    pos_weight_value = n_safe / max(n_crash, 1)

print(f"\n⚙️  pos_weight = {pos_weight_value:.4f}")
print(f"   Minority class: {minority_label}")
if pos_weight_value < 1.0:
    print(f"   ℹ️  pos_weight < 1 means we DOWNWEIGHT crash predictions (crash is majority).")
    print(f"   This is correct: in a class-imbalanced dataset where crash >> safe,")
    print(f"   we want the model to work harder on predicting Safe correctly.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 2.2 — Save the Transformer Dataset
# ══════════════════════════════════════════════════════════════

SAVE_DIR = '/content/drive/MyDrive/Datasets'
np.save(os.path.join(SAVE_DIR, 'transformer_X_train.npy'), transformer_X)
np.save(os.path.join(SAVE_DIR, 'transformer_Y_train.npy'), transformer_Y)
print(f"💾 Saved transformer_X_train.npy ({transformer_X.nbytes / 1e6:.1f} MB)")
print(f"💾 Saved transformer_Y_train.npy ({transformer_Y.nbytes / 1e6:.1f} MB)")

## Transformer Architecture

A tiny Transformer Encoder optimized for edge deployment:
- **Input dim**: 2 (hemo + vent probabilities)
- **Heads**: 2 (one attends to hemo trends, one to vent trends)
- **Layers**: 2 (shallow for zero-latency inference)
- **Pooling**: Average pool across the 60-second sequence
- **Output**: Single probability score via Linear(d_model → 1)

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 3.1 — Positional Encoding
# ══════════════════════════════════════════════════════════════

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# ══════════════════════════════════════════════════════════════
# Step 3.2 — Micro-Transformer Model
# ══════════════════════════════════════════════════════════════

class MicroTransformer(nn.Module):
    """
    Tiny time-series Transformer for conflict resolution.
    Input:  [batch, 60, 2]  — 60 seconds of (hemo_prob, vent_prob)
    Output: [batch]         — raw logit
    """
    def __init__(self, input_dim=2, d_model=64, n_heads=2, n_layers=2,
                 dim_feedforward=128, dropout=0.2, seq_len=60):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len + 10, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, 1),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x.mean(dim=1)
        x = self.classifier(x)
        return x.squeeze(1)


transformer = MicroTransformer().to(device)
total_params = sum(p.numel() for p in transformer.parameters())
trainable = sum(p.numel() for p in transformer.parameters() if p.requires_grad)
print(f"Total parameters  : {total_params:,}")
print(f"Trainable params  : {trainable:,}")

dummy = torch.zeros(2, 60, 2).to(device)
with torch.no_grad():
    out = transformer(dummy)
print(f"Output shape      : {out.shape}  (expected: torch.Size([2]))")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 4.1 — DataLoader Setup (CHRONOLOGICAL SPLIT)
# ══════════════════════════════════════════════════════════════

class ThoughtSequenceDataset(Dataset):
    """Dataset of 60-second AI thought sequences."""
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y).float()
    def __len__(self):
        return len(self.Y)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

BATCH_SIZE = 128
VAL_SPLIT  = 0.20

# ── CHRONOLOGICAL split (not random!) ──
# With 60-second sliding windows that overlap by 59 seconds,
# a random split would contaminate validation with training data.
# Instead, we use the last 20% of time as the validation set.
full_dataset = ThoughtSequenceDataset(transformer_X, transformer_Y)
total_len = len(full_dataset)
split_idx = int(total_len * (1 - VAL_SPLIT))

train_indices = list(range(0, split_idx))
val_indices   = list(range(split_idx, total_len))

train_ds = Subset(full_dataset, train_indices)
val_ds   = Subset(full_dataset, val_indices)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=256, shuffle=False,
                          num_workers=2, pin_memory=True)

# Report split balance
train_labels = transformer_Y[:split_idx]
val_labels   = transformer_Y[split_idx:]
print(f"Train: {len(train_ds):,} samples  ({len(train_loader)} batches)")
print(f"  Safe={int((train_labels==0).sum()):,} ({(train_labels==0).mean():.1%})  "
      f"Crash={int((train_labels==1).sum()):,} ({(train_labels==1).mean():.1%})")
print(f"Val:   {len(val_ds):,} samples  ({len(val_loader)} batches)")
print(f"  Safe={int((val_labels==0).sum()):,} ({(val_labels==0).mean():.1%})  "
      f"Crash={int((val_labels==1).sum()):,} ({(val_labels==1).mean():.1%})")
print(f"\n📌 Split type: CHRONOLOGICAL (train=first {100-int(VAL_SPLIT*100)}%, "
      f"val=last {int(VAL_SPLIT*100)}% of timeline)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 4.2 — Loss, Optimizer, Scheduler
# ══════════════════════════════════════════════════════════════

pos_weight = torch.tensor([pos_weight_value], device=device)
print(f"pos_weight = {pos_weight_value:.4f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

LR           = 1e-3
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.Adam(transformer.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

def run_epoch(loader, training):
    """Run one epoch. Returns (avg_loss, accuracy, auc, sensitivity, specificity)."""
    transformer.train(training)
    total_loss, correct, n = 0.0, 0, 0
    all_logits, all_labels = [], []

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for X_batch, Y_batch in loader:
            X_batch = X_batch.to(device, non_blocking=True)
            Y_batch = Y_batch.to(device, non_blocking=True)

            logits = transformer(X_batch)
            loss = criterion(logits, Y_batch)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(transformer.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * len(Y_batch)
            preds = (logits.sigmoid() > 0.5).long()
            correct += (preds == Y_batch.long()).sum().item()
            n += len(Y_batch)
            all_logits.append(logits.detach().cpu())
            all_labels.append(Y_batch.cpu())

    avg_loss = total_loss / n
    accuracy = correct / n
    all_logits_np = torch.cat(all_logits).numpy()
    all_labels_np = torch.cat(all_labels).numpy()
    all_preds_np  = (1 / (1 + np.exp(-all_logits_np)) > 0.5).astype(int)

    try:
        auc = roc_auc_score(all_labels_np, all_logits_np)
    except ValueError:
        auc = float('nan')

    # Sensitivity (recall on crash class) & Specificity (recall on safe class)
    cm = confusion_matrix(all_labels_np.astype(int), all_preds_np, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0, 0, 0, 0)
    sensitivity = tp / max(tp + fn, 1)  # How many crashes did we catch?
    specificity = tn / max(tn + fp, 1)  # How many safe windows did we NOT false-alarm?

    return avg_loss, accuracy, auc, sensitivity, specificity

print("✅ Training pipeline ready")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 4.3 — Training Loop
# ══════════════════════════════════════════════════════════════

NUM_EPOCHS = 40
PATIENCE   = 10
SAVE_PATH  = '/content/transformer_brain.pth'

best_val_loss = float('inf')
epochs_no_improve = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [],
           'val_acc': [], 'val_auc': [], 'val_sens': [], 'val_spec': []}

print(f"{'Ep':>3}  {'TrLoss':>7}  {'TrAcc':>6}  {'VlLoss':>7}  "
      f"{'VlAcc':>6}  {'AUC':>5}  {'Sens':>5}  {'Spec':>5}  {'LR':>8}")
print("─" * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc, _, _, _                       = run_epoch(train_loader, training=True)
    vl_loss, vl_acc, vl_auc, vl_sens, vl_spec      = run_epoch(val_loader,   training=False)

    scheduler.step(vl_loss)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['val_auc'].append(vl_auc)
    history['val_sens'].append(vl_sens)
    history['val_spec'].append(vl_spec)

    marker = ''
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        epochs_no_improve = 0
        torch.save(transformer.state_dict(), SAVE_PATH)
        marker = ' ✅'
    else:
        epochs_no_improve += 1
        marker = f' ({epochs_no_improve}/{PATIENCE})'

    print(f"{epoch:3d}  {tr_loss:7.4f}  {tr_acc:5.1%}  {vl_loss:7.4f}  "
          f"{vl_acc:5.1%}  {vl_auc:5.3f}  {vl_sens:5.3f}  {vl_spec:5.3f}  "
          f"{current_lr:.1e}{marker}")

    if epochs_no_improve >= PATIENCE:
        print(f"\n⏹  Early stopping at epoch {epoch}.")
        break

print(f"\n✅ Best validation loss: {best_val_loss:.4f}")
print(f"   Weights saved → {SAVE_PATH}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 4.4 — Training Curves
# ══════════════════════════════════════════════════════════════

epochs_ran = len(history['train_loss'])
x_axis = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(x_axis, history['train_loss'], label='Train')
axes[0].plot(x_axis, history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x_axis, history['train_acc'], label='Train')
axes[1].plot(x_axis, history['val_acc'], label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

axes[2].plot(x_axis, history['val_auc'], color='purple', label='Val AUC')
axes[2].set_title('Validation AUC-ROC'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(True, alpha=0.3)
axes[2].set_ylim(0, 1)

axes[3].plot(x_axis, history['val_sens'], color='red', label='Sensitivity (crash recall)')
axes[3].plot(x_axis, history['val_spec'], color='green', label='Specificity (safe recall)')
axes[3].set_title('Sens / Spec'); axes[3].set_xlabel('Epoch')
axes[3].legend(fontsize=8); axes[3].grid(True, alpha=0.3)
axes[3].set_ylim(0, 1)

plt.suptitle('Micro-Transformer Training Curves (v2 — chronological split)', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/transformer_training_curves.png', dpi=150)
plt.show()
print("Saved transformer_training_curves.png")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 4.5 — Final Classification Report on Val Set
# ══════════════════════════════════════════════════════════════

# Load best weights
transformer.load_state_dict(torch.load(SAVE_PATH, map_location=device))
transformer.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, Y_batch in val_loader:
        X_batch = X_batch.to(device)
        logits = transformer(X_batch)
        preds = (logits.sigmoid() > 0.5).long().cpu()
        all_preds.append(preds)
        all_labels.append(Y_batch.long())

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print("═" * 50)
print("  CLASSIFICATION REPORT (Val Set — Chronological)")
print("═" * 50)
print(classification_report(all_labels, all_preds,
                            target_names=['Safe (0)', 'Crash (1)']))

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
print(f"Confusion Matrix:")
print(f"  Predicted:    Safe   Crash")
print(f"  Actual Safe:  {cm[0,0]:>5}   {cm[0,1]:>5}")
print(f"  Actual Crash: {cm[1,0]:>5}   {cm[1,1]:>5}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Step 4.6 — Final Export to Google Drive
# ══════════════════════════════════════════════════════════════

import shutil

DRIVE_SAVE = '/content/drive/MyDrive/Datasets/transformer_brain.pth'
shutil.copy(SAVE_PATH, DRIVE_SAVE)
print(f"\n💾 Exported to {DRIVE_SAVE}")

shutil.copy('/content/transformer_training_curves.png',
            '/content/drive/MyDrive/Datasets/transformer_training_curves.png')
print("💾 Training curves saved to Drive")